# Local BLAST: bacterial and fungal ribosomal markers

This notebook introduces a simple local BLAST workflow. We will create protein databases from two proteomes and search the ribosomal-marker sequences against them.

**Author:** Juan Sebastian Malagón Torres

## Learning objectives

- Understand the difference between a query sequence and a BLAST database.
- Create local BLAST databases with `makeblastdb`.
- Run a search and inspect tabular results.
- Compare the evidence obtained for 16S and 18S markers.
- Distinguish a computational exercise from a biologically valid identification.

> This notebook follows the simple Bash-command style used in the FASTQ preprocessing notebook. It assumes that the notebook is opened from this directory and that the project keeps the following structure:
>
> ```text
> project/
> ├── data/06. Alignment/
> └── material/06. Alignment/
> ```

## 1. Biological context and an important limitation

`organism_1.faa` is the *Escherichia coli* proteome and `organism_2.faa` is a *Saccharomyces* proteome. The `.faa` extension indicates amino-acid sequences.

`ribosomal_markers.faa` contains two nucleotide sequences: a 16S marker and an 18S marker. Although the file extension is `.faa`, these sequences contain DNA/RNA alphabet symbols and are not proteins.

Therefore, the biologically correct comparison would be **BLASTn against nucleotide databases** containing 16S/18S sequences. The supplied proteomes cannot provide a rigorous identification of an rRNA marker. In this practical, we nevertheless run **BLASTx** to demonstrate the local-database workflow and then discuss why the result must be interpreted cautiously.

## 2. Check the input files

The original FASTA files will not be modified.

In [1]:
!pwd
!ls -lh ../../data/06.Alignment/

/home/jmalagont/Documents/courses/javeriana/bioinformatics/material/06.Alignment
total 4.9M
-rw------- 1 jmalagont jmalagont 1.7M Sep  5 23:44 organism_1.faa
-rw------- 1 jmalagont jmalagont 3.2M Sep  5 23:47 organism_2.faa
-rw-r--r-- 1 jmalagont jmalagont 3.1K Sep  5 23:24 ribosomal_markers.faa


In [2]:
# Inspect the first records.
!head -8 ../../data/06.Alignment/organism_1.faa
!head -8 ../../data/06.Alignment/organism_2.faa
!cat ../../data/06.Alignment/ribosomal_markers.faa

>AAC73112.1 thr operon leader peptide [Escherichia coli str. K-12 substr. MG1655]
MKRISTTITTTITITTGNGAG
>AAC73113.1 fused aspartate kinase/homoserine dehydrogenase 1 [Escherichia coli str. K-12 substr. MG1655]
MRVLKFGGTSVANAERFLRVADILESNARQGQVATVLSAPAKITNHLVAMIEKTISGQDALPNISDAERIFAELLTGLAA
AQPGFPLAQLKTFVDQEFAQIKHVLHGISLLGQCPDSINAALICRGEKMSIAIMAGVLEARGHNVTVIDPVEKLLAVGHY
LESTVDIAESTRRIAASRIPADHMVLMAGFTAGNEKGELVVLGRNGSDYSAAVLAACLRADCCEIWTDVDGVYTCDPRQV
PDARLLKSMSYQEAMELSYFGAKVLHPRTITPIAQFQIPCLIKNTGNPQAPGTLIGASRDEDELPVKGISNLNNMAMFSV
SGPGMKGMVGMAARVFAAMSRARISVVLITQSSSEYSISFCVPQSDCVRAERAMQEEFYLELKEGLLEPLAVTERLAIIS
>CAI4043414.1 hypothetical protein SKDI_10G0010 [Saccharomyces kudriavzevii IFO 1802]
MHSLEPPRSKKRIHLIAAVKAVKSIKPFRTTLRYDEAITYNKSNEEKEKYTEAYHKEVNQLLKMNAWETDKYYDRNSMGS
KNMISSVLVFNKKRDGTHKARFVARGDIQHPDTYDPGMQSNTVHHYALMTSLSLALDNDYYVTQLDVSSAYLYADIKEEL
YIRPPPHLGLNNKLLSLKKSLYGLKQSGANWYETIKSYLIKQCGMDEVRGWSCVFKNSQVTICLFVDDMILFSKDLKANK
KVIANLRIKKKKKLKP
>CAI4043416.1 hypothetical protein SKDI

## 3. Install or verify BLAST+

The commands below use the BLAST+ command-line tools. If BLAST+ is not installed, run the installation command in the terminal, not inside the analysis cells.

```bash
conda install -n bioinformatics -c bioconda -c conda-forge blast
```

The two programs used here are `makeblastdb` (database construction) and `blastx` (translate a nucleotide query in six reading frames and search a protein database).

In [3]:
!blastx -version
!makeblastdb -version

blastx: 2.17.0+
 Package: blast 2.17.0, build Aug 22 2026 11:52:50
makeblastdb: 2.17.0+
 Package: blast 2.17.0, build Aug 22 2026 11:52:50


## 4. Create folders for the results

The database files and BLAST output are kept in `results/`.

In [4]:
!mkdir -p results/blastdb/organism_1
!mkdir -p results/blastdb/organism_2
!mkdir -p results/blast_results

## 5. Build one protein database for each organism

`-in` is the input FASTA file, `-dbtype prot` says that the sequences are proteins, and `-out` is the prefix for the database files.

In [5]:
!makeblastdb -in ../../data/06.Alignment/organism_1.faa -dbtype prot -parse_seqids -out results/blastdb/organism_1/ecoli
!makeblastdb -in ../../data/06.Alignment/organism_2.faa -dbtype prot -parse_seqids -out results/blastdb/organism_2/yeast



Building a new DB, current time: 09/06/2026 00:24:06
New DB name:   /home/jmalagont/Documents/courses/javeriana/bioinformatics/material/06.Alignment/results/blastdb/organism_1/ecoli
New DB title:  ../../data/06.Alignment/organism_1.faa
Sequence type: Protein
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 4300 sequences in 0.120065 seconds.




Building a new DB, current time: 09/06/2026 00:24:06
New DB name:   /home/jmalagont/Documents/courses/javeriana/bioinformatics/material/06.Alignment/results/blastdb/organism_2/yeast
New DB title:  ../../data/06.Alignment/organism_2.faa
Sequence type: Protein
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 5607 sequences in 0.14706 seconds.




In [6]:
!ls -lh results/blastdb/organism_1
!ls -lh results/blastdb/organism_2

total 2.3M
-rw-r--r-- 1 jmalagont jmalagont 228K Sep  6 00:24 ecoli.pdb
-rw-r--r-- 1 jmalagont jmalagont 577K Sep  6 00:24 ecoli.phr
-rw-r--r-- 1 jmalagont jmalagont  34K Sep  6 00:24 ecoli.pin
-rw-r--r-- 1 jmalagont jmalagont  500 Sep  6 00:24 ecoli.pjs
-rw-r--r-- 1 jmalagont jmalagont  17K Sep  6 00:24 ecoli.pog
-rw-r--r-- 1 jmalagont jmalagont  80K Sep  6 00:24 ecoli.pos
-rw-r--r-- 1 jmalagont jmalagont  51K Sep  6 00:24 ecoli.pot
-rw-r--r-- 1 jmalagont jmalagont 1.3M Sep  6 00:24 ecoli.psq
-rw-r--r-- 1 jmalagont jmalagont  16K Sep  6 00:24 ecoli.ptf
-rw-r--r-- 1 jmalagont jmalagont  17K Sep  6 00:24 ecoli.pto
total 4.0M
-rw-r--r-- 1 jmalagont jmalagont 312K Sep  6 00:24 yeast.pdb
-rw-r--r-- 1 jmalagont jmalagont 718K Sep  6 00:24 yeast.phr
-rw-r--r-- 1 jmalagont jmalagont  44K Sep  6 00:24 yeast.pin
-rw-r--r-- 1 jmalagont jmalagont  500 Sep  6 00:24 yeast.pjs
-rw-r--r-- 1 jmalagont jmalagont  22K Sep  6 00:24 yeast.pog
-rw-r--r-- 1 jmalagont jmalagont 115K Sep  6 00:24 yeast.pos
-r

## 6. Run BLASTx with the 16S marker

The output format is tabular (`6`). The columns are query, subject, percent identity, alignment length, mismatches, gap openings, query start/end, subject start/end, E-value, and bit score.

A low E-value and a high bit score are evidence of a better alignment. They are not, by themselves, proof of species identity.

In [24]:
!blastx -query ../../data/06.Alignment/ribosomal_markers.faa -db results/blastdb/organism_1/ecoli -out results/blast_results/markers_vs_ecoli.tsv -max_target_seqs 1 -outfmt 6 -num_threads 2 
!blastx -query ../../data/06.Alignment/ribosomal_markers.faa -db results/blastdb/organism_2/yeast -out results/blast_results/markers_vs_yeast.tsv  -max_target_seqs 1 -outfmt 6 -num_threads 2

## 7. Compare the best hit for each marker

The command below prints the hit with the highest bit score for each query and each database. Empty output means that no alignment passed the E-value threshold.

In [27]:
!echo '--- 16S/18S vs E. coli proteome ---'
!sort -k1,1 -k12,12gr results/blast_results/markers_vs_ecoli.tsv | head -20
!echo '--- 16S/18S vs Saccharomyces proteome ---'
!sort -k1,1 -k12,12gr results/blast_results/markers_vs_yeast.tsv | head -20

--- 16S/18S vs E. coli proteome ---
16S	AAC74660.4	25.301	83	49	4	1124	1339	277	357	3.2	27.3
18S	AAC76085.1	34.091	44	27	1	1596	1727	3	44	6.2	24.6
--- 16S/18S vs Saccharomyces proteome ---
16S	CAI4054556.1	25.217	115	73	3	971	1309	237	340	0.68	30.4
18S	CAI4055024.1	35.714	56	32	3	1607	1765	262	316	0.66	30.8


## 8. Interpretation questions

1. Does the 16S marker produce a convincing hit against the *E. coli* proteome?
2. Does the 18S marker produce a convincing hit against the *Saccharomyces* proteome?
3. What happens when a marker has no significant hit?
4. Why is translating an rRNA sequence with BLASTx biologically inappropriate?
5. Which BLAST program and database type should be used for a real 16S/18S identification?

**Expected conclusion:** the proteome-vs-rRNA experiment is useful for learning the mechanics of local BLAST, but it should not be used as the final evidence that one marker belongs to a bacterium or a fungus. For identification, use `blastn` with curated nucleotide databases containing bacterial 16S and fungal 18S sequences, or use a dedicated rRNA classifier.

## Reproducibility checklist

Record the input files, BLAST+ version, database type, database FASTA files, E-value threshold, number of target sequences, and output format. Keep the original FASTA files unchanged.